# 🔢➕ Aula 2 — NumPy (complementar)

**Módulo DS-PY-004 · Técnicas de Programação I (Python)** · CAIXAVERSO FC6 · Analista de Dados II

Este notebook é o complemento da aula: eu **não** o apresento ao vivo. Aqui eu guardo o que ficou de fora do notebook principal (`02-numpy-principal.ipynb`) e deixo mais exercícios para você praticar no seu ritmo, com as respostas escondidas.

Se algum assunto da aula ficou confuso, volte ao principal primeiro. Se quiser ir além, este é o lugar.

## 🗺️ Roteiro

1. 🧬 **Tipos de dados e memória:** `dtype`, `astype`, `nbytes`, e por que `float32` e `int8` pedem cuidado
2. 📦 **Mais dimensões:** tensor (3D), `vstack` e `hstack`
3. 📐 **Funções matemáticas:** arredondar, juros compostos, `inf` e percentis
4. 🔄 **Broadcasting completo:** as regras e as tabelas cruzadas
5. 🧰 **Quatro ferramentas extras:** `np.where`, `np.clip`, `np.argsort` e `np.unique`
6. 🧮 **Álgebra linear:** `np.linalg.solve` e o erro da matriz singular
7. ⏱️ **Laço × vetorização,** medindo o tempo
8. 🧾 **Resumo** e 🏋️ **exercícios por área** (financeiro, comercial e RH)

## ⚙️ Configuração

Recriamos as **mesmas bases** do notebook principal (mesmas sementes, então os números são idênticos).

| Variável | Formato | O que é | Unidade |
|---|---|---|---|
| `saldos` | `(500,)` | Saldo em conta de 500 clientes (negativo = cheque especial) | R$ |
| `vendas_mensais` | `(12, 6)` | Vendas de 6 agências (colunas) em 12 meses (linhas) | R$ mil |
| `salarios` | `(40,)` | Salário de 40 colaboradores | R$ |

In [1]:
# Esta célula importa as bibliotecas e mostra a versão do NumPy.
import time
import numpy as np

print("NumPy:", np.__version__)

NumPy: 2.5.4


In [2]:
# Esta célula recria as três bases com as mesmas sementes do notebook principal.
rng = np.random.default_rng(42)
saldos = rng.normal(loc=5000, scale=3000, size=500).round(2)

vendas_mensais = np.random.default_rng(7).integers(80, 200, size=(12, 6))
salarios = np.random.default_rng(3).integers(2500, 12000, size=40)

## 1. 🧬 Tipos de dados e memória

Todo array guarda **um único tipo** de número, no atributo `dtype`.

| `dtype` | Significa | Exemplo |
|---|---|---|
| `int64` | inteiro | quantidade de transações |
| `float64` | decimal | saldo em reais |
| `bool` | verdadeiro/falso | cliente inadimplente? |
| `<U...` | texto | nome da agência |

In [3]:
# Esta célula mostra o tipo dos saldos e das vendas mensais.
print("saldos        :", saldos.dtype)
print("vendas_mensais:", vendas_mensais.dtype)

saldos        : float64
vendas_mensais: int64


O NumPy escolhe o tipo sozinho. Se você mistura tipos, ele **converte tudo para o mais "largo"**: inteiro com decimal vira decimal, número com texto vira texto.

In [4]:
# Esta célula mistura inteiros e um decimal.
np.array([100, 250, 80.5])

array([100. , 250. ,  80.5])

In [5]:
# Esta célula mistura números e um texto.
np.array([100, 250, "isento"])

array(['100', '250', 'isento'], dtype='<U21')

Na segunda, tudo virou texto (`'100'`, `'250'`, `'isento'`), e com texto não dá para fazer conta.

### `astype`: converter o tipo

`astype` devolve um **novo array** com o tipo pedido. Cuidado ao converter decimal para inteiro:

In [6]:
# Esta célula converte os 5 primeiros saldos para inteiro.
primeiros = saldos[:5]
print(primeiros)
print(primeiros.astype(int))

[5914.15 1880.05 7251.35 7821.69 -853.11]
[5914 1880 7251 7821 -853]


`astype(int)` **corta** a parte decimal, sem arredondar: `7821.69` virou `7821`. Para arredondar antes, use `np.round(primeiros).astype(int)`.

### Quanto cada array ocupa na memória

`itemsize` é quanto cada valor ocupa (em bytes) e `nbytes` é o total.

In [7]:
# Esta célula mostra o espaço ocupado pelos saldos.
print("itemsize:", saldos.itemsize, "bytes por valor")
print("nbytes  :", saldos.nbytes, "bytes no total")

itemsize: 8 bytes por valor
nbytes  : 4000 bytes no total


Cada saldo (`float64`) ocupa 8 bytes, e são 500 saldos: 4.000 bytes. Com **1 milhão** de saldos a diferença entre os tipos aparece:

In [8]:
# Esta célula compara o espaço de 1 milhão de saldos em três tipos decimais.
um_milhao = np.random.default_rng(0).normal(5000, 3000, size=1_000_000)

for tipo in ["float64", "float32", "float16"]:
    print(tipo, "->", um_milhao.astype(tipo).nbytes / 1_000_000, "MB")

float64 -> 8.0 MB
float32 -> 4.0 MB
float16 -> 2.0 MB


Menos bytes por número, menos memória. Mas existe uma troca: **menos precisão**. Veja um saldo grande com centavos:

In [9]:
# Esta célula guarda o mesmo saldo em float64 e em float32.
saldo_exato = 123_456_789.12

print("float64:", np.float64(saldo_exato))
print("float32:", np.float32(saldo_exato))

float64: 123456789.12
float32: 1.2345679e+08


O `float32` perdeu os centavos (guarda só uns 7 dígitos). **Para dinheiro, use `float64`** (ou trabalhe em centavos inteiros).

### Inteiros têm limite: o *overflow*

Cada tipo inteiro tem uma faixa. O `np.iinfo` mostra:

In [10]:
# Esta célula mostra o menor e o maior valor de cada tipo inteiro.
for tipo in [np.int8, np.int16, np.int32, np.int64]:
    info = np.iinfo(tipo)
    print(f"{tipo.__name__:6}: de {info.min:,} a {info.max:,}")

int8  : de -128 a 127
int16 : de -32,768 a 32,767
int32 : de -2,147,483,648 a 2,147,483,647
int64 : de -9,223,372,036,854,775,808 a 9,223,372,036,854,775,807


O `int8` só vai de -128 a 127. Passou disso, o número **"dá a volta"**, e o pior: sem avisar. Exemplo com um contador de transações:

In [11]:
# Esta célula dobra um contador guardado como int8.
contador = np.array([100, 120, 90], dtype=np.int8)
contador * 2

array([-56, -16, -76], dtype=int8)

`100 × 2` deveria dar 200, mas deu **-56** (200 - 256). É um erro silencioso, pior que uma exceção. A correção é usar um tipo maior:

In [12]:
# Esta célula refaz a conta com int16.
contador.astype(np.int16) * 2

array([200, 240, 180], dtype=int16)

### `astype(int)` com valor faltante

Lembra do `np.nan`? Ele só existe em decimal. Veja o que acontece ao converter para inteiro:

In [13]:
# Esta célula converte para inteiro um array que tem nan.
com_falta = np.array([100.0, np.nan, 250.0])
com_falta.astype(int)

C:\Users\davih\AppData\Local\Temp\ipykernel_18120\290891412.py:3: RuntimeWarning: invalid value encountered in cast
  com_falta.astype(int)


array([                 100, -9223372036854775808,                  250])

Não deu erro, mas o NumPy avisou (`RuntimeWarning: invalid value encountered in cast`) e o `nan` virou o número absurdo `-9223372036854775808`. Não existe "inteiro faltante". **Trate os `nan` antes de converter.**

## 2. 📦 Mais dimensões: tensor, `vstack` e `hstack`

### Tensor (3 dimensões)

Vetor tem 1 dimensão, tabela tem 2. Com **3 dimensões**, o array se chama **tensor**: pense em várias tabelas empilhadas, como **2 semanas × 3 dias × 2 agências**.

In [14]:
# Esta célula cria um tensor: 2 semanas, cada uma com 3 dias e 2 agências.
tensor = np.array([
    [[120, 85], [150, 90], [110, 95]],    # semana 1
    [[130, 80], [140, 70], [125, 100]],   # semana 2
])

print("ndim :", tensor.ndim)
print("shape:", tensor.shape)
print("size :", tensor.size)

ndim : 3
shape: (2, 3, 2)
size : 12


O `shape` `(2, 3, 2)` lê-se da esquerda para a direita: 2 semanas, 3 dias, 2 agências. O total de elementos é 2 × 3 × 2 = 12.

In [15]:
# Esta célula mostra a semana 1 inteira e o valor do dia 3 da semana 2 na agência 1.
print(tensor[0])
print(tensor[1, 2, 1])

[[120  85]
 [150  90]
 [110  95]]
100


In [16]:
# Esta célula soma os dias e as agências, obtendo o total de cada semana.
tensor.sum(axis=(1, 2))

array([650, 645])

Semana 1: R$ 650 mil; semana 2: R$ 645 mil. O `axis=(1, 2)` colapsa as duas últimas dimensões de uma vez.

### `vstack` e `hstack`: juntar arrays

- `np.vstack` empilha **verticalmente** (acrescenta **linhas**);
- `np.hstack` junta **horizontalmente** (acrescenta **colunas**).

Vamos acrescentar 2 meses novos e 1 agência nova à tabela de vendas.

In [17]:
# Esta célula cria 2 meses novos (2 x 6) e 1 agência nova (12 x 1).
rng_novos = np.random.default_rng(8)
meses_novos = rng_novos.integers(80, 200, size=(2, 6))
agencia_nova = rng_novos.integers(80, 200, size=(12, 1))

In [18]:
# Esta célula acrescenta os meses (vstack) e a agência (hstack).
com_14_meses = np.vstack([vendas_mensais, meses_novos])
com_7_agencias = np.hstack([vendas_mensais, agencia_nova])

print("vstack:", com_14_meses.shape)
print("hstack:", com_7_agencias.shape)

vstack: (14, 6)
hstack: (12, 7)


`(12, 6)` virou `(14, 6)` (+2 linhas) e `(12, 7)` (+1 coluna). Para juntar, a dimensão que **não cresce** precisa coincidir. **Vou provocar o erro:**

In [19]:
# Esta célula tenta juntar lado a lado tabelas com número de linhas diferente (vai dar erro).
np.hstack([vendas_mensais, meses_novos])

ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 12 and the array at index 1 has size 2

Como ler: `ValueError: all the input array dimensions except for the concatenation axis must match exactly`. Para juntar **colunas**, o número de **linhas** tem de ser igual (12 ≠ 2).

## 3. 📐 Funções matemáticas, `inf` e percentis

### Funções que agem em cada valor

| Função | O que faz |
|---|---|
| `np.abs` | valor absoluto (tira o sinal) |
| `np.round(x, n)` | arredonda para `n` casas |
| `np.floor` / `np.ceil` | arredonda para baixo / para cima |
| `np.sqrt` | raiz quadrada |
| `np.exp` / `np.log` | exponencial / logaritmo natural |

In [20]:
# Esta célula aplica abs, round e floor nos 5 primeiros saldos.
amostra = saldos[:5]
print("original:", amostra)
print("abs     :", np.abs(amostra))
print("round   :", np.round(amostra))
print("floor   :", np.floor(amostra))

original: [5914.15 1880.05 7251.35 7821.69 -853.11]
abs     : [5914.15 1880.05 7251.35 7821.69  853.11]
round   : [5914. 1880. 7251. 7822. -853.]
floor   : [5914. 1880. 7251. 7821. -854.]


O `abs` tirou o sinal do saldo negativo; o `round` mandou `7821.69` para `7822.` (diferente do `astype(int)`, que cortou); o `floor` sempre arredonda para baixo (`-853.11` virou `-854.`).

### Juros compostos sem laço

Aplicar R$ 10.000 a 2% ao mês. O montante após `n` meses é `capital * (1 + taxa) ** n`. Passando um **array de meses**, calculamos os 12 de uma vez:

In [21]:
# Esta célula calcula o montante de cada um dos 12 meses.
capital = 10_000
meses = np.arange(1, 13)

montante = capital * 1.02 ** meses
montante.round(2)

array([10200.  , 10404.  , 10612.08, 10824.32, 11040.81, 11261.62,
       11486.86, 11716.59, 11950.93, 12189.94, 12433.74, 12682.42])

No mês 12 o montante chega a **R$ 12.682,42**. O crescimento acelera porque os juros incidem sobre os juros anteriores.

### Dividir por zero: `inf`

No NumPy, dividir por zero **não derruba o programa**: gera `inf` (infinito) e um aviso. Exemplo: saldo dividido pelo limite de crédito, em que um cliente tem limite 0.

In [22]:
# Esta célula divide o saldo pelo limite de crédito (há limites iguais a zero).
saldo_cliente = np.array([3000.0, 1500.0, 0.0, 800.0])
limite_credito = np.array([5000.0, 0.0, 0.0, 2000.0])

razao = saldo_cliente / limite_credito
razao

C:\Users\davih\AppData\Local\Temp\ipykernel_18120\64895688.py:5: RuntimeWarning: divide by zero encountered in divide
  razao = saldo_cliente / limite_credito
C:\Users\davih\AppData\Local\Temp\ipykernel_18120\64895688.py:5: RuntimeWarning: invalid value encountered in divide
  razao = saldo_cliente / limite_credito


array([0.6, inf, nan, 0.4])

Os `RuntimeWarning` avisam do problema, mas a conta segue: `1500 / 0` deu `inf` e `0 / 0` deu `nan`. Para separar o que é número válido:

In [23]:
# Esta célula marca quais resultados são números válidos.
np.isfinite(razao)

array([ True, False, False,  True])

### Percentis

O **percentil** diz abaixo de qual valor está certa porcentagem dos dados. Os de 25, 50 e 75 são os **quartis**.

In [24]:
# Esta célula calcula os quartis dos saldos.
np.percentile(saldos, [25, 50, 75])

array([2999.2475, 5009.635 , 6762.1825])

Um quarto dos clientes tem saldo abaixo de cerca de R$ 3.000, metade abaixo de R$ 5.010 (a mediana) e três quartos abaixo de R$ 6.762.

### Média × mediana quando há valores extremos

Entra um cliente "VIP" com R$ 5 milhões. Veja o efeito em cada medida:

In [25]:
# Esta célula compara média e mediana antes e depois de incluir o cliente VIP.
com_vip = np.append(saldos, 5_000_000)

print("média  :", saldos.mean().round(2), "->", com_vip.mean().round(2))
print("mediana:", np.median(saldos).round(2), "->", np.median(com_vip).round(2))

média  : 4960.62 -> 14930.76
mediana: 5009.64 -> 5019.02


A **média** quase triplicou, e a **mediana** mal se mexeu. Quando existem valores muito fora do padrão (*outliers*), a mediana descreve melhor o cliente "típico".

## 4. 🔄 Broadcasting completo

No principal vimos o essencial. Aqui, a regra completa para **dois arrays**:

1. Alinhe os formatos pela **direita**.
2. O que tem menos dimensões ganha `1` à **esquerda**.
3. Em cada posição, as dimensões devem ser **iguais** ou uma delas deve ser **1** (a de tamanho 1 é "esticada").
4. O resultado tem, em cada posição, o **maior** dos dois valores.
5. Se alguma posição quebrar a regra 3, vem um `ValueError`.

A função `np.broadcast_shapes` aplica a regra e mostra o formato do resultado, **antes de calcular**:

In [26]:
# Esta célula testa três combinações de formatos que funcionam.
print(np.broadcast_shapes((12, 6), (6,)))
print(np.broadcast_shapes((12, 6), (12, 1)))
print(np.broadcast_shapes((3, 1), (1, 4)))

(12, 6)
(12, 6)
(3, 4)


Os resultados são `(12, 6)`, `(12, 6)` e `(3, 4)`. O último é especial: uma **coluna** com uma **linha** gera uma **tabela cruzada** (todas as combinações). Agora uma combinação que não funciona. **Vou provocar o erro:**

In [27]:
# Esta célula testa uma combinação que não encaixa (vai dar erro).
np.broadcast_shapes((12, 6), (12,))

ValueError: shape mismatch: objects cannot be broadcast to a single shape.  Mismatch is between arg 0 with shape (12, 6) and arg 1 with shape (12,).

Como ler: `ValueError: shape mismatch: objects cannot be broadcast to a single shape`. Alinhando pela direita, o `6` fica contra o `12`: diferentes, e nenhum é 1.

| Formato A | Formato B | O que acontece | Resultado |
|---|---|---|---|
| `(12, 6)` | `(6,)` | B vira `(1, 6)` | `(12, 6)` |
| `(12, 6)` | `(12, 1)` | o `1` é esticado | `(12, 6)` |
| `(3, 1)` | `(1, 4)` | os dois são esticados | `(3, 4)` |
| `(12, 6)` | `(12,)` | B vira `(1, 12)`: `6 ≠ 12` | **erro** |

### Aplicação: tabela cruzada de simulação

Quanto rende R$ 1.000 em cada combinação de **taxa** (3 valores, em coluna) e **prazo** (4 valores, em linha)?

In [28]:
# Esta célula monta a tabela cruzada: 3 taxas (linhas) x 4 prazos (colunas).
taxas = np.array([0.01, 0.02, 0.03])
prazos = np.array([6, 12, 24, 36])

simulacao = 1000 * (1 + taxas[:, None]) ** prazos[None, :]
simulacao.round(2)

array([[1061.52, 1126.83, 1269.73, 1430.77],
       [1126.16, 1268.24, 1608.44, 2039.89],
       [1194.05, 1425.76, 2032.79, 2898.28]])

Cada **linha** é uma taxa (1%, 2%, 3% ao mês) e cada **coluna** é um prazo (6, 12, 24 e 36 meses). A última célula, 3% por 36 meses, chega a quase R$ 2.900. Tudo sem laço.

## 5. 🧰 Quatro ferramentas extras

### `np.where`: um "se" para o array inteiro

`np.where(condição, valor_se_verdadeiro, valor_se_falso)` escolhe, posição por posição.

In [29]:
# Esta célula classifica os 8 primeiros clientes como devedor ou em dia.
np.where(saldos[:8] < 0, "devedor", "em dia")

array(['em dia', 'em dia', 'em dia', 'em dia', 'devedor', 'em dia',
       'em dia', 'em dia'], dtype='<U7')

O 5º cliente, com saldo de -853,11, aparece como `devedor`.

### `np.clip`: limitar a uma faixa

Uma comissão de 2% sobre cada venda, mas **no mínimo R$ 2 mil e no máximo R$ 3 mil**:

In [30]:
# Esta célula calcula a comissão bruta e a limitada ao intervalo de 2 a 3.
comissao_bruta = vendas_mensais * 0.02
comissao_final = np.clip(comissao_bruta, 2, 3)

print("sem limite:", comissao_bruta[0])
print("com limite:", comissao_final[0])

sem limite: [3.86 3.1  3.24 3.74 2.98 3.46]
com limite: [3.   3.   3.   3.   2.98 3.  ]


Quem estava abaixo de 2 subiu para 2, e quem passava de 3 caiu para 3. Os valores do meio ficaram como estavam.

### `np.argsort`: a ordem, e não os valores

`np.sort` devolve os valores ordenados. `np.argsort` devolve as **posições** que ordenariam o array, o que permite ordenar **os nomes junto**.

In [31]:
# Esta célula ordena as agências, da que mais vendeu para a que menos vendeu.
nomes = np.array(["Centro", "Norte", "Sul", "Leste", "Oeste", "Porto"])
total_por_agencia = vendas_mensais.sum(axis=0)

ordem = np.argsort(total_por_agencia)[::-1]    # [::-1] inverte a ordem

for nome, total in zip(nomes[ordem], total_por_agencia[ordem]):
    print(f"{nome:7} R$ {total} mil")

Leste   R$ 1848 mil
Centro  R$ 1828 mil
Sul     R$ 1800 mil
Oeste   R$ 1731 mil
Porto   R$ 1611 mil
Norte   R$ 1503 mil


Usamos a mesma `ordem` para reorganizar os nomes e os totais, e cada nome continua junto do seu valor.

### `np.unique`: valores distintos e quantas vezes aparecem

In [32]:
# Esta célula conta quantos clientes há em cada situação.
situacao = np.where(saldos < 0, "devedor", "em dia")
valores, contagens = np.unique(situacao, return_counts=True)

for valor, contagem in zip(valores, contagens):
    print(valor, contagem)

devedor 22
em dia 478


São 22 devedores e 478 em dia, o mesmo número que contamos no principal.

## 6. 🧮 Álgebra linear com `np.linalg`

Um problema comum: **descobrir tarifas** a partir do que cada agência arrecadou. Sabemos quantos serviços cada agência cobrou (matriz `qtd`) e quanto cada uma arrecadou (`receita`). Queremos as tarifas `x` tais que `qtd @ x = receita`.

In [33]:
# Esta célula cria as quantidades de serviços por agência e a receita de cada uma.
qtd = np.array([
    [100, 50, 20],   # agência A: TED, saque, extrato
    [ 80, 60, 30],   # agência B
    [120, 40, 30],   # agência C
])
receita = np.array([974.0, 836.0, 1136.0])

In [34]:
# Esta célula resolve o sistema e descobre a tarifa de cada serviço.
tarifas = np.linalg.solve(qtd, receita)
tarifas

array([8.5, 2. , 1.2])

As tarifas são R$ 8,50 (TED), R$ 2,00 (saque) e R$ 1,20 (extrato). Para conferir, multiplicamos de volta:

In [35]:
# Esta célula confere se as tarifas reproduzem a receita.
np.allclose(qtd @ tarifas, receita)

True

`True`: as tarifas encontradas reproduzem a receita.

| Função | O que faz |
|---|---|
| `np.linalg.solve(m, b)` | resolve `m @ x = b` |
| `np.linalg.det(m)` | determinante (se for 0, não há solução única) |
| `np.linalg.inv(m)` | matriz inversa |

> 💡 Para resolver sistemas, prefira `solve` a `inv(m) @ b`: é mais rápido e mais preciso.

### Erro comum: matriz singular

Se uma agência tiver exatamente o **dobro** dos serviços de outra, as linhas são redundantes e o sistema não tem solução única. **Vou provocar o erro:**

In [36]:
# Esta célula usa uma matriz com uma linha igual ao dobro de outra (vai dar erro).
qtd_redundante = np.array([
    [100, 50, 20],
    [200, 100, 40],
    [120, 40, 30],
])
np.linalg.solve(qtd_redundante, receita)

LinAlgError: Singular matrix

Como ler: `LinAlgError: Singular matrix`. "Singular" quer dizer determinante zero: não há inversa. Na prática, falta informação **independente** nos dados.

## 7. ⏱️ Laço × vetorização, medindo o tempo

No principal usamos o `%timeit`. Aqui vamos medir de outro jeito, com `time.perf_counter()`, um cronômetro de alta precisão. 🔧 **Função pronta**: ela executa uma função e devolve o resultado junto com o tempo gasto.

In [37]:
# Função pronta: executa uma função e devolve (resultado, segundos).
def medir(funcao, *args):
    inicio = time.perf_counter()
    resultado = funcao(*args)
    return resultado, time.perf_counter() - inicio

A conta: **juros compostos de 12 meses** (2% ao mês) sobre **1 milhão de saldos**, de três jeitos.

In [38]:
# Esta célula cria 1 milhão de saldos e define as três versões da conta.
saldos_grandes = np.random.default_rng(0).normal(5000, 3000, size=1_000_000)
lista_grande = saldos_grandes.tolist()

def com_for(lista):
    resultado = []
    for saldo in lista:
        resultado.append(saldo * 1.02 ** 12)
    return resultado

def com_list_comprehension(lista):
    return [saldo * 1.02 ** 12 for saldo in lista]

def com_numpy(array):
    return array * 1.02 ** 12

In [39]:
# Esta célula mede o tempo de cada versão e mostra quantas vezes o NumPy é mais rápido.
r_for, t_for = medir(com_for, lista_grande)
_, t_lc = medir(com_list_comprehension, lista_grande)
r_np, t_np = medir(com_numpy, saldos_grandes)

print(f"for                : {t_for:.4f} s")
print(f"list comprehension : {t_lc:.4f} s")
print(f"NumPy              : {t_np:.4f} s")
print(f"NumPy é {t_for / t_np:.0f}x mais rápido que o for")

for                : 0.0475 s
list comprehension : 0.0432 s
NumPy              : 0.0016 s
NumPy é 30x mais rápido que o for


No meu computador, o `for` levou cerca de **0,045 s**, a *list comprehension* cerca de **0,040 s** e o NumPy cerca de **0,0015 s**: o NumPy foi cerca de **30 vezes** mais rápido. A *list comprehension* ajuda um pouco, mas continua longe do array.

Os valores mudam a cada execução e de computador para computador; o que importa é a ordem de grandeza. E o resultado é o mesmo?

In [40]:
# Esta célula confere se o for e o NumPy chegaram ao mesmo resultado.
np.allclose(r_for, r_np)

True

`True`. O ganho de velocidade não muda o resultado.

> 💡 **Cuidados ao medir:** repita a medição, meça só a conta (sem a criação dos dados) e, quando o ganho for pequeno, prefira o código mais legível.

## 🧾 Resumo

| Comando | Para que serve |
|---|---|
| `a.dtype`, `a.astype(tipo)` | ver / converter o tipo (`astype(int)` corta os decimais) |
| `a.itemsize`, `a.nbytes` | bytes por valor / total |
| `np.iinfo(tipo)` | faixa de um tipo inteiro (cuidado com *overflow*) |
| `a.ndim`, `a.shape` em 3D | tensor: ex.: (semanas, dias, agências) |
| `np.vstack`, `np.hstack` | acrescenta linhas / colunas |
| `np.abs`, `np.round`, `np.floor`, `np.sqrt`, `np.exp`, `np.log` | funções em cada valor |
| `np.isfinite` | separa números válidos de `inf` e `nan` |
| `np.percentile(a, [25, 50, 75])` | percentis (quartis) |
| `np.broadcast_shapes(f1, f2)` | prevê o formato do broadcasting |
| `a[:, None]`, `a[None, :]` | transforma vetor em coluna / linha |
| `np.where(cond, a, b)` | "se" para o array inteiro |
| `np.clip(a, mínimo, máximo)` | limita valores a uma faixa |
| `np.argsort(a)` | posições que ordenam o array |
| `np.unique(a, return_counts=True)` | valores distintos e contagem |
| `np.linalg.solve(m, b)` | resolve sistemas |
| `time.perf_counter()` | cronômetro |

**Armadilhas:** `float32` perde centavos; `int8` dá a volta sem avisar; `astype(int)` corta os decimais; `solve` falha em matriz singular.

## 🏋️ Exercícios extras

Resolva na célula vazia e confira na resposta escondida. Use `saldos`, `vendas_mensais` e `salarios`.

---

### 💰 Financeiro

**Exercício 1 — Memória.** Quantos MB ocupam 10 milhões de saldos em `float64`? E em `float32`? Quanto se economiza?

In [41]:
# sua resposta aqui

In [42]:
# Resposta
n = 10_000_000
mb_64 = n * np.dtype("float64").itemsize / 1_000_000
mb_32 = n * np.dtype("float32").itemsize / 1_000_000

print(f"float64: {mb_64:.0f} MB | float32: {mb_32:.0f} MB | economia: {mb_64 - mb_32:.0f} MB")

float64: 80 MB | float32: 40 MB | economia: 40 MB


**Exercício 2 — Faixas de saldo.** Classifique cada cliente como `"devedor"` (saldo abaixo de 0), `"baixo"` (de 0 a 2.000) ou `"normal"` (acima de 2.000) e conte quantos há em cada grupo.

*Dica: aninhe dois `np.where`.*

In [43]:
# sua resposta aqui

In [44]:
# Resposta
grupo = np.where(saldos < 0, "devedor",
                 np.where(saldos <= 2000, "baixo", "normal"))

valores, contagens = np.unique(grupo, return_counts=True)
print({str(v): int(c) for v, c in zip(valores, contagens)}, "| total:", contagens.sum())

{'baixo': 57, 'devedor': 22, 'normal': 421} | total: 500


**Exercício 3 — Dobrar o capital.** Um cliente aplica R$ 10.000 a 3% ao mês. Em que mês o montante passa de R$ 20.000? Calcule os 36 primeiros meses de uma vez.

*Dica: `argmax` em uma máscara devolve a posição do primeiro `True`.*

In [45]:
# sua resposta aqui

In [46]:
# Resposta
meses = np.arange(1, 37)
montante = 10_000 * 1.03 ** meses

mes_que_dobrou = (montante >= 20_000).argmax() + 1
print("o capital dobra no mês:", mes_que_dobrou)
print("montante nesse mês    :", montante[mes_que_dobrou - 1].round(2))

o capital dobra no mês: 24
montante nesse mês    : 20327.94


---

### 🛒 Comercial

**Exercício 4 — Cenários de crescimento.** Calcule o total do ano de cada agência e simule o resultado em 3 cenários: **0%, 5% e 10%** de crescimento. O resultado deve ser uma tabela `(6, 3)`.

In [47]:
# sua resposta aqui

In [48]:
# Resposta
total_ano = vendas_mensais.sum(axis=0)
cenarios = np.array([0.00, 0.05, 0.10])

projecao = total_ano[:, None] * (1 + cenarios[None, :])
print(projecao.shape)
projecao.round(1)

(6, 3)


array([[1828. , 1919.4, 2010.8],
       [1503. , 1578.2, 1653.3],
       [1800. , 1890. , 1980. ],
       [1848. , 1940.4, 2032.8],
       [1731. , 1817.6, 1904.1],
       [1611. , 1691.6, 1772.1]])

**Exercício 5 — Comissão com piso e teto.** A comissão é 2% de cada venda mensal, com **piso de 2** e **teto de 3** (R$ mil). Quantas comissões ficaram no piso e quantas no teto?

In [49]:
# sua resposta aqui

In [50]:
# Resposta
bruta = vendas_mensais * 0.02
final = np.clip(bruta, 2, 3)

print("no piso :", (bruta <= 2).sum())
print("no teto :", (bruta >= 3).sum())
print("total   :", final.size)

no piso : 11
no teto : 31
total   : 72


**Exercício 6 — Ranking.** Mostre as **3 agências com maior venda no ano** e as **3 com menor**, com os números das agências e os totais.

In [51]:
# sua resposta aqui

In [52]:
# Resposta
total_ano = vendas_mensais.sum(axis=0)
ordem = np.argsort(total_ano)

menores = ordem[:3]
maiores = ordem[-3:][::-1]

print("maiores:", maiores, total_ano[maiores])
print("menores:", menores, total_ano[menores])

maiores: [3 0 2] [1848 1828 1800]
menores: [1 5 4] [1503 1611 1731]


---

### 👥 RH

**Exercício 7 — Quartis salariais.** Calcule os quartis (25, 50 e 75) dos salários. Quantos colaboradores ganham **acima do 3º quartil**?

In [53]:
# sua resposta aqui

In [54]:
# Resposta
q1, q2, q3 = np.percentile(salarios, [25, 50, 75])
print("quartis:", q1, q2, q3)
print("acima do 3º quartil:", (salarios > q3).sum())

quartis: 4948.5 6922.5 9205.0
acima do 3º quartil: 10


**Exercício 8 — Preço dos itens.** Três kits de material de escritório têm preço conhecido. O kit A tem 2 cadernos, 1 caneta e 1 pasta e custa R$ 20,20. O kit B tem 1 caderno, 3 canetas e 2 pastas e custa R$ 16,90. O kit C tem 3 cadernos, nenhuma caneta e 5 pastas e custa R$ 31,50. Qual o preço de cada item?

In [55]:
# sua resposta aqui

In [56]:
# Resposta
kits = np.array([
    [2, 1, 1],
    [1, 3, 2],
    [3, 0, 5],
])
precos_kits = np.array([20.20, 16.90, 31.50])

caderno, caneta, pasta = np.linalg.solve(kits, precos_kits)
print(f"caderno: {caderno:.2f} | caneta: {caneta:.2f} | pasta: {pasta:.2f}")

caderno: 8.50 | caneta: 2.00 | pasta: 1.20


**Exercício 9 (desafio) — Laço × vetorização.** Aplique um reajuste de 5% só aos salários **abaixo de R$ 8.000**, em uma base de **2 milhões** de colaboradores. Faça com `for` e com NumPy, meça o tempo de cada um e confirme que os resultados são iguais.

*Dica: reaproveite a função `medir` e use `np.where`.*

In [57]:
# Esta célula gera 2 milhões de salários para o desafio.
salarios_grande = np.random.default_rng(5).integers(2500, 12000, size=2_000_000)

# sua resposta aqui

In [58]:
# Resposta
def reajuste_com_for(array):
    saida = []
    for salario in array.tolist():
        saida.append(salario * 1.05 if salario < 8000 else salario)
    return saida

def reajuste_com_numpy(array):
    return np.where(array < 8000, array * 1.05, array)

r_for, t_for = medir(reajuste_com_for, salarios_grande)
r_np, t_np = medir(reajuste_com_numpy, salarios_grande)

print(f"for: {t_for:.3f} s | NumPy: {t_np:.3f} s | NumPy {t_for / t_np:.0f}x mais rápido")
print("resultados iguais:", np.allclose(r_for, r_np))

for: 0.175 s | NumPy: 0.011 s | NumPy 17x mais rápido
resultados iguais: True


## 🚀 Próxima aula

Na **Aula 3** começamos o **pandas**: o NumPy com **nomes e rótulos** nas linhas e colunas. A base será a de 30 mil clientes de cartão de crédito. Tudo o que você praticou aqui continua valendo.